In [1]:
from bs4 import BeautifulSoup as bs
import requests as req
from pprint import pprint
import json

### 💻 實作練習：ETtoday 財經新聞爬蟲與 JSON 資料儲存

#### 🎯 任務目標
本練習將結合我們所學的 `requests` 與 `BeautifulSoup` 模組，向「ETtoday 財經新聞」發送請求，萃取網頁中的新聞標題、內容摘要、連結與時間。最後，將清洗後的結構化資料序列化，並輸出為一份標準的 JSON 檔案。

#### 🏗️ 實作流程

[發送 HTTP GET 請求] 

    ↓

[建立 BeautifulSoup 實例] 

    ↓

[DOM 節點尋找與遍歷] 

    ↓

[資料清洗與建構 Dictionary] 

    ↓
    
[JSON 序列化寫入實體檔案]

In [ ]:
# --- 1. 目標設定與變數初始化 ---
url = 'https://finance.ettoday.net/focus/775'

# --- 2. 發送 HTTP 請求與建立解析物件 ---
# 發送 GET 請求至目標網址，並將伺服器回傳的結果儲存於 res 物件中
res = req.get(url)

# bs(...) : 正在進行「實例化 (Instantiation)」
# 將純文字的 HTML 源碼 (res.text) 傳入 BeautifulSoup 類別，
# 並指定 'lxml' 作為解析器，生成可供搜尋與遍歷的 soup 物件
soup = bs(res.text, 'lxml')

# 初始化一個空的 List，用於暫存後續萃取出的每一筆新聞資料
new_list = []

# --- 3. DOM 節點遍歷與資料清洗 ---
# find_all 回傳一個 ResultSet，包含所有符合 <a class="piece clearfix"> 的節點
news = soup.find_all('a',class_='piece clearfix')

# 透過 for 迴圈遍歷每一個新聞節點
for new in news:
    # 建立字典結構 (Dictionary) 以鍵值對 (Key-Value Pair) 儲存單筆新聞特徵
    new_contain ={
        # 萃取屬性 'title'
        'title' : new['title'].replace('\u3000', ' '),
        # 定位 <p> 標籤並擷取其內部文字作為新聞摘要/內容
        'contain': new.p.text,
        # 萃取屬性 'href' 獲取新聞的超連結
        'url' : new['href'],
        # 在當前節點下，進一步向下搜尋 <p class="date"> 並擷取時間文字
        'time': new.find('p',class_='date').text
    }
    # 將建構好的單筆新聞字典，附加至整體資料的陣列中
    new_list.append(new_contain)

# 於終端機印出結果，作為程式開發階段的除錯與確認 (Debugging / Validation)
for n in new_list:
    print(n)

# --- 4. JSON 序列化與檔案輸出 ---
# 宣告輸出的檔案名稱，確保副檔名為 .json 以符合資料格式
file_name = 'ettoday-fin-news.json'


# 使用 Context Manager (with 語句) 開啟檔案，確保 I/O 資源會被正確釋放
# 'w' 模式代表寫入 (Write)，若檔案不存在則建立，若存在則覆寫
# encoding='utf-8' 確保以 UTF-8 編碼寫入檔案

# json.dump 直接將資料結構序列化並寫入檔案物件 f
# ensure_ascii = False 確保中文正常顯示，不被轉換為 Unicode 編碼
# indent = 4 增加縮排，使 JSON 結構清晰

with open(file_name, 'w', encoding='utf-8') as f:
    json.dump(new_list, f, ensure_ascii=False, indent=4)


{'title': '為了結婚買房 離開5年舒適圈「賺高薪後悔了」嘆：職場環境更重要', 'contain': '一名網友表示，由於準備結婚、買房，每月還得負擔3萬多元房貸，離開待了5年的職場，選擇薪資更高的職缺，原本期待能換來更好的生活品質，沒想到新工作的職場氣氛、同事相處與工作壓力，讓他吃不消，開始懷念過去那份「錢比較少但人比較好」的工作。對此，網友感嘆「長大後才知道，錢不是最重要的」。', 'url': 'https://finance.ettoday.net/news/3155616', 'time': '2小時前'}
{'title': '日經近逼6萬點 一文看日股ETF單月漲幅4猛將', 'contain': '日經225指數全球人工智慧（AI）浪潮與地緣政治緊張局勢緩解激勵下，上周五（24日）以59716.18點作收，距離6萬點位，不到300點，帶動國內日股ETF強勁漲勢，包括台新日本半導體(00951)、中信日本半導體(00954)、元大日經225(00661)、國泰日經225(00657)等四檔四月以來皆大漲二位數，其中，台新日本半導體(00951)四月以來大漲近22%、今年來大漲35%。\r\n\r\n', 'url': 'https://finance.ettoday.net/news/3155646', 'time': '3小時前'}
{'title': '川普揚言奪回5成晶片市場 謝金河：半導體業美中角力升高', 'contain': '美國總統川普4月23日揚言美國很快將取得全球50%的晶片市場占有率，警告晶片業者如果不到美國設廠生產，最快就會面臨非常高的關稅。對此，財信傳媒董事長謝金河在臉書上以「世界的重心在半導體」為題，撰文表示，美伊戰事爆發以來，造成油價上漲，而資本市場重心在半導體產業上，川普對半導體談話內話，顯示接下來美中角力最激烈的板塊一定是在半導體。', 'url': 'https://finance.ettoday.net/news/3155559', 'time': '5小時前'}
{'title': 'HOYA BIT奪全球首張ISO 14068-1 成碳中和加密交易所第一家', 'contain': '台灣加密貨幣交易所 HOYA BIT（禾亞數位科技）宣布，通過英國標準協會（BSI）驗證的 ISO 14068-1「碳

### 💻 實作練習：ETtoday 財經新聞爬蟲與 CSV 資料儲存

In [ ]:
import pandas as pd
url = 'https://finance.ettoday.net/focus/775'

res = req.get(url)
#soup = bs(...) : 正在進行「實例化 (Instantiation)」
soup = bs(res.text, 'lxml')

new_list = []

news = soup.find_all('a',class_='piece clearfix')
for new in news:
    new_contain ={
        'title' : new['title'].replace('\u3000', ' '),
        'contain': new.p.text,
        'url' : new['href'],
        'time': new.find('p',class_='date').text
    }
    new_list.append(new_contain)

for n in new_list:
    print(n)

# 將 List of Dictionaries 轉換為 DataFrame
df = pd.DataFrame(new_list)

# 將 DataFrame 輸出為 CSV 檔案
csv_file_name = 'ettoday-fin-news.csv'

df.to_csv(csv_file_name, index=False ,encoding='utf-8-sig')

{'title': '為了結婚買房 離開5年舒適圈「賺高薪後悔了」嘆：職場環境更重要', 'contain': '一名網友表示，由於準備結婚、買房，每月還得負擔3萬多元房貸，離開待了5年的職場，選擇薪資更高的職缺，原本期待能換來更好的生活品質，沒想到新工作的職場氣氛、同事相處與工作壓力，讓他吃不消，開始懷念過去那份「錢比較少但人比較好」的工作。對此，網友感嘆「長大後才知道，錢不是最重要的」。', 'url': 'https://finance.ettoday.net/news/3155616', 'time': '3小時前'}
{'title': '日經近逼6萬點 一文看日股ETF單月漲幅4猛將', 'contain': '日經225指數全球人工智慧（AI）浪潮與地緣政治緊張局勢緩解激勵下，上周五（24日）以59716.18點作收，距離6萬點位，不到300點，帶動國內日股ETF強勁漲勢，包括台新日本半導體(00951)、中信日本半導體(00954)、元大日經225(00661)、國泰日經225(00657)等四檔四月以來皆大漲二位數，其中，台新日本半導體(00951)四月以來大漲近22%、今年來大漲35%。\r\n\r\n', 'url': 'https://finance.ettoday.net/news/3155646', 'time': '3小時前'}
{'title': '川普揚言奪回5成晶片市場 謝金河：半導體業美中角力升高', 'contain': '美國總統川普4月23日揚言美國很快將取得全球50%的晶片市場占有率，警告晶片業者如果不到美國設廠生產，最快就會面臨非常高的關稅。對此，財信傳媒董事長謝金河在臉書上以「世界的重心在半導體」為題，撰文表示，美伊戰事爆發以來，造成油價上漲，而資本市場重心在半導體產業上，川普對半導體談話內話，顯示接下來美中角力最激烈的板塊一定是在半導體。', 'url': 'https://finance.ettoday.net/news/3155559', 'time': '5小時前'}
{'title': 'HOYA BIT奪全球首張ISO 14068-1 成碳中和加密交易所第一家', 'contain': '台灣加密貨幣交易所 HOYA BIT（禾亞數位科技）宣布，通過英國標準協會（BSI）驗證的 ISO 14068-1「碳

### 💻 實作練習：ETtoday 財經新聞爬蟲一次爬娶七頁

In [11]:
base_url = 'https://finance.ettoday.net/focus/775'
page_list = [' ','/2','/3','/4','/5','/6','/7']
new_list = []
for page in page_list:
    url = base_url + page
    res = req.get(url)
    print(f"正在爬取{url}")
    #soup = bs(...) : 正在進行「實例化 (Instantiation)」
    soup = bs(res.text, 'lxml')
    news = soup.find_all('a',class_='piece clearfix')
    for new in news:
        new_contain ={
            'title' : new['title'].replace('\u3000', ' '),
            'contain': new.p.text,
            'url' : new['href'],
            'time': new.find('p',class_='date').text
        }
        new_list.append(new_contain)

    for n in new_list:
        print(n)

    # 檔名，副檔名要記得為 .json
    file_name = 'ettoday-fin-news_more.json'

    # json.dump 直接將資料結構序列化並寫入檔案物件 f
    # ensure_ascii = False 確保中文正常顯示，不被轉換為 Unicode 編碼
    # indent = 4 增加縮排，使 JSON 結構清晰
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(new_list, f, ensure_ascii=False, indent=4)


正在爬取https://finance.ettoday.net/focus/775 
{'title': '為了結婚買房 離開5年舒適圈「賺高薪後悔了」嘆：職場環境更重要', 'contain': '一名網友表示，由於準備結婚、買房，每月還得負擔3萬多元房貸，離開待了5年的職場，選擇薪資更高的職缺，原本期待能換來更好的生活品質，沒想到新工作的職場氣氛、同事相處與工作壓力，讓他吃不消，開始懷念過去那份「錢比較少但人比較好」的工作。對此，網友感嘆「長大後才知道，錢不是最重要的」。', 'url': 'https://finance.ettoday.net/news/3155616', 'time': '3小時前'}
{'title': '日經近逼6萬點 一文看日股ETF單月漲幅4猛將', 'contain': '日經225指數全球人工智慧（AI）浪潮與地緣政治緊張局勢緩解激勵下，上周五（24日）以59716.18點作收，距離6萬點位，不到300點，帶動國內日股ETF強勁漲勢，包括台新日本半導體(00951)、中信日本半導體(00954)、元大日經225(00661)、國泰日經225(00657)等四檔四月以來皆大漲二位數，其中，台新日本半導體(00951)四月以來大漲近22%、今年來大漲35%。\r\n\r\n', 'url': 'https://finance.ettoday.net/news/3155646', 'time': '3小時前'}
{'title': '川普揚言奪回5成晶片市場 謝金河：半導體業美中角力升高', 'contain': '美國總統川普4月23日揚言美國很快將取得全球50%的晶片市場占有率，警告晶片業者如果不到美國設廠生產，最快就會面臨非常高的關稅。對此，財信傳媒董事長謝金河在臉書上以「世界的重心在半導體」為題，撰文表示，美伊戰事爆發以來，造成油價上漲，而資本市場重心在半導體產業上，川普對半導體談話內話，顯示接下來美中角力最激烈的板塊一定是在半導體。', 'url': 'https://finance.ettoday.net/news/3155559', 'time': '5小時前'}
{'title': 'HOYA BIT奪全球首張ISO 14068-1 成碳中和加密交易所第一家', 'contain': '台灣加密貨幣交易所 HOYA B